In [1]:
import time
import random
import pandas as pd
import numpy as np
import polars as pl

In [2]:
input_params = pd.DataFrame(np.random.random(size=(50000000, 4)),
                            columns=['param_a', 'param_b', 'param_c', 'param_d'])
input_params

,param_a,param_b,param_c,param_d
0,0.648617,0.321176,0.839502,0.253110
1,0.185445,0.380637,0.616461,0.476327
2,0.706604,0.959392,0.867478,0.983852
3,0.803859,0.075646,0.540660,0.510797
4,0.259248,0.278624,0.682906,0.483885
...,...,...,...,...
49999995,0.681439,0.693412,0.290986,0.591613
49999996,0.353911,0.051687,0.276539,0.448045
49999997,0.788036,0.476394,0.852945,0.115951
49999998,0.404577,0.932189,0.594752,0.970193


In [4]:
# Create two separate dataframes
df1 = pl.DataFrame(data=np.random.random(size=(50000000, 4)),
                            schema=['param_a', 'param_b', 'param_c', 'param_d'])

df2 = pl.DataFrame(data=np.random.random(size=(50000000, 4)),
                            schema=['param_a', 'param_b', 'param_c', 'param_d'])

# Define the lazy operations for each dataframe
# The .lazy() method converts the DataFrame to a LazyFrame
lazy_df1 = df1.lazy().with_columns(
    (pl.col("param_a") * 10).alias("new_param_a")
).select(
    "new_param_a", "param_b"
)

lazy_df2 = df2.lazy().with_columns(
    (pl.col("param_a") + 100).alias("new_param_a")
).select(
    "new_param_a", "param_b"
)

# Polars' optimizer will handle the execution
result_df1 = lazy_df1.collect()
result_df2 = lazy_df2.collect()

print("Result 1:")
print(result_df1)

print("\nResult 2:")
print(result_df2)

Result 1:
shape: (50_000_000, 2)
┌─────────────┬──────────┐
│ new_param_a ┆ param_b  │
│ ---         ┆ ---      │
│ f64         ┆ f64      │
╞═════════════╪══════════╡
│ 6.131907    ┆ 0.674533 │
│ 9.118543    ┆ 0.960586 │
│ 1.48282     ┆ 0.120732 │
│ 9.803458    ┆ 0.574416 │
│ 2.004995    ┆ 0.459159 │
│ …           ┆ …        │
│ 6.865858    ┆ 0.583372 │
│ 7.344174    ┆ 0.773997 │
│ 4.688716    ┆ 0.409039 │
│ 8.003972    ┆ 0.903286 │
│ 4.105895    ┆ 0.371192 │
└─────────────┴──────────┘

Result 2:
shape: (50_000_000, 2)
┌─────────────┬──────────┐
│ new_param_a ┆ param_b  │
│ ---         ┆ ---      │
│ f64         ┆ f64      │
╞═════════════╪══════════╡
│ 100.318622  ┆ 0.681982 │
│ 100.087067  ┆ 0.254925 │
│ 100.246538  ┆ 0.201232 │
│ 100.267508  ┆ 0.946404 │
│ 100.991588  ┆ 0.798453 │
│ …           ┆ …        │
│ 100.483648  ┆ 0.942644 │
│ 100.265479  ┆ 0.669757 │
│ 100.851923  ┆ 0.196546 │
│ 100.703475  ┆ 0.452242 │
│ 100.350776  ┆ 0.762345 │
└─────────────┴──────────┘


In [22]:
def pl_process_data(data: pl.DataFrame, common_columns, new_data_cols = None, new_data_print_cols = None, desired_order = None):
    data = data.drop_nans()

    if new_data_cols:
        # Map cadenza column names to their print names
        rename_dict_new = dict(zip(new_data_cols, new_data_print_cols))

        # Select only common columns (columns that occur in both training and new data)
        short_dict = {key:value for key, value in rename_dict_new.items() if value in common_columns}

        # Select common columns, rename them and order them identically to the training data
        data = data[list(short_dict.keys())].rename(short_dict)[desired_order]

    # Select numeric columns
    numeric_df = data.select(cs.numeric())

    # Handle empty dataframe
    if any([dim==0 for dim in df1.shape]):
        print("No numerical columns in the dataframe")
        return data.head(0)
    
    # Sample to speed up report generation
    return numeric_df.sample(fraction=0.15, seed=17)

In [93]:
pl_process_data(df1, ["A", "B"], new_data_cols =["param_a", "param_b", "param_c", "param_d"], new_data_print_cols =["A", "B", "C", "D"], desired_order = ["B", "A"])

B,A
f64,f64
0.897341,0.6072
0.775272,0.869352
0.369081,0.897666
0.867238,0.311106
0.893875,0.309798
…,…
0.591538,0.35489
0.068177,0.754016
0.785315,0.73356


In [ ]:
def process_data(ref_data, new_data, new_data_cols, new_data_print_cols, common_columns, npartitions=4):
    # Initialize local dask cluster
    with LocalCluster(processes=False, n_workers=1, threads_per_worker=4, memory_limit='4GB') as cluster:
        with Client(cluster) as client:
            print(f"Dask dashboard available at: {client.dashboard_link}")
            
            # Drop nas
            ref_data.dropna(inplace=True)
            new_data.dropna(inplace=True)

            print("New data before:", new_data)

            # Process ref data
            ref_data = ref_data[common_columns]
            print("ref-data passed")
            
            # Process new data
            rename_dict_new = dict(zip(new_data_cols, new_data_print_cols))
            print("dict passed", rename_dict_new)
            short_dict = {key:value for key, value in rename_dict_new.items() if value in common_columns}

            print("short-dict passed")
            new_data = new_data[short_dict.keys()].rename(columns=short_dict)

            print("New data after:", new_data)
            
            # Re-order new data
            print("Old column order: ", new_data)
            desired_order = ref_data.columns.tolist()
            new_data = new_data[desired_order]

            print("New column order: ", new_data)

            print("New data processed")

            # Sample datasets
            frac = ref_data.shape[0] / new_data.shape[0] 

            if frac < 1: 
                new_data = new_data.sample(frac=frac)
            else:
                ref_data= ref_data.sample(frac=1/frac)

            new_data = get_samples(new_data)
            ref_data = get_samples(ref_data)

    
    return new_data, ref_data

In [4]:
def map_to_def(dataFrame: pl.DataFrame, id_col: list = [], datetime_cols: list = []):
    """
    Processes a DataFrame to identify column types and prepares a DataDefinition for Evidently.
    """
    df = dataFrame.clone()
    df.columns = [str(col) for col in df.columns]

    numerical_cols = df.select(cs.numeric()).columns
    categorical_cols = df.select(cs.string()).columns

    remove_set = set()

    id_column_name = id_col[0] if id_col and len(id_col) > 0 else None

    # Ensure special columns are not in the feature lists
    if id_column_name:
        remove_set.add(id_column_name)

    if datetime_cols:
        for col in datetime_cols:
            if col in df.columns:
                print(f"Converting {col} to datetime")
                df[col] = pd.to_datetime(df[col], errors='coerce')
                remove_set.add(col)

    numerical_cols = [col for col in numerical_cols if col not in remove_set]
    categorical_cols = [col for col in categorical_cols if col not in remove_set]


    return df, DataDefinition(
        id_column=id_column_name,
        numerical_columns=numerical_cols,
        categorical_columns=categorical_cols,
        datetime_columns=datetime_cols,
    )

In [49]:
new_dataset = Dataset.from_pandas(df1.to_pandas())

In [50]:
ref_dataset = Dataset.from_pandas(df2.to_pandas())

In [51]:
report = Report([DataDriftPreset(method="psi")], include_tests="True")
reps = report.run(ref_dataset, new_dataset)

: 

: 

In [10]:
df3, deff = map_to_def(df1)

In [8]:
df3

(shape: (50_000_000, 4)
 ┌──────────┬──────────┬──────────┬──────────┐
 │ param_a  ┆ param_b  ┆ param_c  ┆ param_d  │
 │ ---      ┆ ---      ┆ ---      ┆ ---      │
 │ f64      ┆ f64      ┆ f64      ┆ f64      │
 ╞══════════╪══════════╪══════════╪══════════╡
 │ 0.976935 ┆ 0.044881 ┆ 0.645864 ┆ 0.896393 │
 │ 0.064132 ┆ 0.945164 ┆ 0.194237 ┆ 0.643121 │
 │ 0.152669 ┆ 0.018531 ┆ 0.238074 ┆ 0.385754 │
 │ 0.294358 ┆ 0.704269 ┆ 0.366327 ┆ 0.972725 │
 │ 0.961516 ┆ 0.940169 ┆ 0.025125 ┆ 0.904063 │
 │ …        ┆ …        ┆ …        ┆ …        │
 │ 0.241948 ┆ 0.949511 ┆ 0.814757 ┆ 0.029957 │
 │ 0.019579 ┆ 0.782137 ┆ 0.569879 ┆ 0.707238 │
 │ 0.897738 ┆ 0.436334 ┆ 0.733504 ┆ 0.903912 │
 │ 0.244006 ┆ 0.361219 ┆ 0.024492 ┆ 0.155908 │
 │ 0.270183 ┆ 0.911878 ┆ 0.08018  ┆ 0.322865 │
 └──────────┴──────────┴──────────┴──────────┘,
 DataDefinition(id_column=None, timestamp=None, service_columns=None, numerical_columns=['param_a', 'param_b', 'param_c', 'param_d'], categorical_columns=[], text_columns=None,

In [11]:
deff

DataDefinition(id_column=None, timestamp=None, service_columns=None, numerical_columns=['param_a', 'param_b', 'param_c', 'param_d'], categorical_columns=[], text_columns=None, datetime_columns=[], classification=None, regression=None, llm=None, numerical_descriptors=[], categorical_descriptors=[], test_descriptors=None, ranking=None, special_columns=[])

In [101]:
import orjson

In [ ]:
split_dict = {
    "index": list(range(df1.height)),
    "columns": df1.columns,
    "data": df1.rows()
}

In [ ]:
payload ={
            "reference_data": {
                "data": split_dict
            }
}

[{'param_a': 0.005229298278324523,
  'param_b': 0.9689835063602011,
  'param_c': 0.9811437971042184,
  'param_d': 0.11734060584838912},
 {'param_a': 0.2299111463813841,
  'param_b': 0.33004078593946906,
  'param_c': 0.034077663381764545,
  'param_d': 0.07501983607259599},
 {'param_a': 0.05027251919891551,
  'param_b': 0.5590337432299116,
  'param_c': 0.035425059243321955,
  'param_d': 0.9442024045414281},
 {'param_a': 0.48784377690530023,
  'param_b': 0.9518447598347947,
  'param_c': 0.5598791963351698,
  'param_d': 0.5606854329560925},
 {'param_a': 0.08819955314455807,
  'param_b': 0.28712017834448755,
  'param_c': 0.7973034025657019,
  'param_d': 0.6856352923145596},
 {'param_a': 0.008147223986190943,
  'param_b': 0.383500267894168,
  'param_c': 0.23647657560701918,
  'param_d': 0.6829504303427221},
 {'param_a': 0.48041648278940885,
  'param_b': 0.4722573310367253,
  'param_c': 0.9639388364281413,
  'param_d': 0.39547761686089256},
 {'param_a': 0.7681313542634759,
  'param_b': 0.1529

In [4]:
import numpy as np
import polars as pl
import timeit

df1 = pl.DataFrame(data=np.random.random(size=(50000000, 2)),
                            schema=['param_a', 'param_b'])

dicts = timeit.timeit("df1.to_dicts()", globals=globals(), number=1)

pds = timeit.timeit("df1.to_pandas().to_dict(orient='records')", globals=globals(), number=1)

print(f"Time for df1.to_dicts(): {dicts:.2f} seconds")
print(f"Time for df1.to_pandas().to_dict(orient='records'): {pds:.2f} seconds")

Time for df1.to_dicts(): 194.98 seconds
Time for df1.to_pandas().to_dict(orient='records'): 120.11 seconds


In [10]:
import polars as pl
import orjson
import timeit
import numpy as np
df1 = pl.DataFrame(data=np.random.random(size=(10_000_000, 3)),
                   schema=['param_a', 'param_b', 'param_c'])

In [13]:
import polars.selectors as cs

In [16]:
df1.filter(
   pl.all_horizontal(pl.col(pl.Float32, pl.Float64).is_not_nan())
)

param_a,param_b,param_c
f64,f64,f64
0.894099,0.56414,0.564784
0.42498,0.938006,0.259766
0.8316,0.235255,0.292592
0.49731,0.127116,0.810088
0.439324,0.335182,0.568473
…,…,…
0.78439,0.830629,0.870104
0.82822,0.995906,0.530249
0.597317,0.549892,0.090473


In [14]:
df1.filter(pl.all_horizontal(cs.float().is_not_nan()))

param_a,param_b,param_c
f64,f64,f64
0.894099,0.56414,0.564784
0.42498,0.938006,0.259766
0.8316,0.235255,0.292592
0.49731,0.127116,0.810088
0.439324,0.335182,0.568473
…,…,…
0.78439,0.830629,0.870104
0.82822,0.995906,0.530249
0.597317,0.549892,0.090473


In [11]:
# Option 1: The standard Polars way (can be slow due to float conversion)
def polars_to_dicts_method():
    return df1.to_dicts()

# Option 2: The Polars-to-Pandas method (surprisingly faster in your case)
def polars_to_pandas_to_dicts_method():
    return df1.to_pandas().to_dict(orient="records")

def to_pandas_to_json_method():
    data = df1.to_pandas().to_dict(orient="records")
    return orjson.dumps(data)

# Option 3: The recommended, fastest method using orjson and to_dicts()
def fast_json_serialization_orjson():
    data = df1.to_dicts()
    return orjson.dumps(data)

# Run the benchmarks
print("Benchmarking Polars to dicts directly:")
time_to_dicts = timeit.timeit(polars_to_dicts_method, number=1)
print(f"Time: {time_to_dicts:.2f} seconds")
print("---")

print("Benchmarking Polars -> Pandas -> dicts:")
time_to_pandas_dicts = timeit.timeit(polars_to_pandas_to_dicts_method, number=1)
print(f"Time: {time_to_pandas_dicts:.2f} seconds")
print("---")

print("Benchmarking Pandas to dict + orjson serialization:")
time_pandas_with_orjson = timeit.timeit(to_pandas_to_json_method, number=1)
print(f"Time: {time_pandas_with_orjson:.2f} seconds")
print("---")

print("Benchmarking Polars to dicts + orjson serialization:")
time_with_orjson = timeit.timeit(fast_json_serialization_orjson, number=1)
print(f"Time: {time_with_orjson:.2f} seconds")

Benchmarking Polars to dicts directly:
Time: 14.36 seconds
---
Benchmarking Polars -> Pandas -> dicts:
Time: 13.56 seconds
---
Benchmarking Pandas to dict + orjson serialization:
Time: 15.92 seconds
---
Benchmarking Polars to dicts + orjson serialization:
Time: 14.29 seconds


In [4]:
import pandas as pd
import numpy as np

In [102]:
# df = pd.DataFrame(np.random.random(size=(5, 4)),
#                             columns=['param_a', 'param_b', 'param_c', 'param_d'])
df = pd.DataFrame({'A': [1.0, .2, .3, .4], 'B': [5, 4, 3, 2]})

In [28]:
lambda row: get_row(df, row)

<function __main__.<lambda>(row)>

In [29]:
df.itertuples()

In [90]:
def get_row(df, row):
    print(row)
    row_data = [df.at[row, col] for col in df.columns]
    return row_data

In [103]:
row = df.iloc[0]

In [104]:
a = df.iloc[0]
a

A    1.0
B    5.0
Name: 0, dtype: float64

In [105]:
for b in a:
    print(b)

1.0
5.0


In [91]:
df.select_dtypes("Int64")

,B
0,5
